# Visionary — Stanford Cars fine-tuning (Kaggle / Colab GPU)

Runs the **same scripts** used for the submitted model, so results are reproducible:
`prepare_data.py` → `train.py` → `train_classical.py` → `export_onnx.py` → `evaluate.py` → `write_report.py`.

Enable a GPU (P100/T4) and Internet in the notebook settings.

In [ ]:
!git clone https://github.com/Sreeram5678/Visionary.git
%cd Visionary
!pip install -q timm pyarrow onnx onnxruntime onnxscript scikit-image
!pip install -q --no-deps ultralytics ultralytics-thop

## 1. Download Stanford Cars (with boxes) and prepare crops + stratified train/val split

In [ ]:
%%bash
mkdir -p data/stanford_cars data/tanganke
for f in train-00000-of-00002 train-00001-of-00002 test-00000-of-00002 test-00001-of-00002; do
  wget -q -O data/stanford_cars/$f.parquet https://huggingface.co/datasets/naufalso/stanford_cars/resolve/main/data/$f.parquet
  wget -q -O data/tanganke/$f.parquet https://huggingface.co/datasets/tanganke/stanford_cars/resolve/main/data/$f.parquet
done
python scripts/prepare_data.py

## 2. Fine-tune (default model: ConvNeXt-Tiny @224, ~15 epochs)
For the larger model use e.g. `--backbone convnext_small.fb_in22k_ft_in1k --img 320 --bs 16 --accum 2 --name convnext_small_320`.

In [ ]:
!python scripts/train.py --backbone convnext_tiny.fb_in22k_ft_in1k --img 224 --epochs 15 --bs 32 --workers 2 --name convnext_tiny_224

## 3. Classical baseline, INT8 export, full evaluation, report

In [ ]:
!python scripts/train_classical.py
!python scripts/export_onnx.py --ckpt weights/convnext_tiny_224.pt
!python scripts/evaluate.py --ckpt weights/convnext_tiny_224.pt
!python scripts/write_report.py --default convnext_tiny_224
!head -40 eval_P5_T20.md